In [0]:
from pyspark.sql import functions as F

TABELA_SILVER = "workspace.silver.inep_superior_cursos"

TABELA_GOLD_UF = (
    "workspace.gold.fato_inep_educacao_uf_anual"
)

TABELA_GOLD_BRASIL = (
    "workspace.gold.fato_inep_educacao_brasil_anual"
)

ANOS_PROCESSAMENTO = [2021, 2022, 2023, 2024]

df_inep = (
    spark.table(TABELA_SILVER)
    .filter(F.col("ano_censo").isin(ANOS_PROCESSAMENTO))
)

print("Origem:", TABELA_SILVER)
print("Gold UF:", TABELA_GOLD_UF)
print("Gold Brasil:", TABELA_GOLD_BRASIL)

Origem: workspace.silver.inep_superior_cursos
Gold UF: workspace.gold.fato_inep_educacao_uf_anual
Gold Brasil: workspace.gold.fato_inep_educacao_brasil_anual


In [0]:
df_analise_dimensao = (
    df_inep
    .groupBy(
        "ano_censo",
        "tipo_dimensao",
        "tipo_modalidade_ensino"
    )
    .agg(
        F.count("*").alias("registros"),

        F.sum(
            F.when(
                F.col("sigla_uf").isNull()
                | (F.trim(F.col("sigla_uf")) == ""),
                1
            ).otherwise(0)
        ).alias("registros_sem_uf"),

        F.countDistinct("sigla_uf").alias("ufs_distintas"),

        F.sum("quantidade_cursos").alias("cursos"),
        F.sum("quantidade_vagas").alias("vagas"),
        F.sum("quantidade_inscritos").alias("inscritos"),
        F.sum("quantidade_ingressantes").alias("ingressantes"),
        F.sum("quantidade_matriculas").alias("matriculas"),
        F.sum("quantidade_concluintes").alias("concluintes")
    )
    .orderBy(
        "ano_censo",
        "tipo_dimensao",
        "tipo_modalidade_ensino"
    )
)

display(df_analise_dimensao)

ano_censo,tipo_dimensao,tipo_modalidade_ensino,registros,registros_sem_uf,ufs_distintas,cursos,vagas,inscritos,ingressantes,matriculas,concluintes
2021,1,1,35750,0,27,35482,5942212,8644865,1467717,5270750,842184
2021,2,2,401088,0,27,0,0,0,2476016,3713646,484777
2021,3,2,7624,7624,0,7620,16736850,6475962,0,0,0
2021,4,2,324,324,0,0,0,0,1358,2724,364
2022,1,1,36089,0,27,35774,5658590,8811102,1656401,5113182,803801
2022,2,2,527328,1,27,0,0,0,3098768,4328116,483335
2022,3,2,9207,9207,0,9186,17171895,8176586,0,0,0
2022,4,2,395,395,0,0,0,0,1788,2818,499
2023,1,1,35739,0,27,35410,5440999,8627123,1679790,5063936,783505
2023,2,2,624836,4,27,0,0,0,3312712,4910515,590782


In [0]:
df_analise_ead = (
    df_inep
    .filter(F.col("tipo_dimensao").isin(2, 3))
    .groupBy("ano_censo", "tipo_dimensao")
    .agg(
        F.count("*").alias("registros"),
        F.countDistinct("codigo_curso").alias("cursos_distintos"),
        F.countDistinct("codigo_ies").alias("ies_distintas"),
        F.sum("quantidade_cursos").alias("quantidade_cursos"),
        F.sum("quantidade_vagas").alias("quantidade_vagas"),
        F.sum("quantidade_ingressantes").alias("ingressantes"),
        F.sum("quantidade_matriculas").alias("matriculas"),
        F.sum("quantidade_concluintes").alias("concluintes")
    )
    .orderBy("ano_censo", "tipo_dimensao")
)

display(df_analise_ead)

ano_censo,tipo_dimensao,registros,cursos_distintos,ies_distintas,quantidade_cursos,quantidade_vagas,ingressantes,matriculas,concluintes
2021,2,401088,7592,559,0,0,2476016,3713646,484777
2021,3,7624,7624,559,7620,16736850,0,0,0
2022,2,527328,9154,644,0,0,3098768,4328116,483335
2022,3,9207,9207,645,9186,17171895,0,0,0
2023,2,624836,10509,723,0,0,3312712,4910515,590782
2023,3,10576,10576,725,10554,19180966,0,0,0
2024,2,673756,11255,791,0,6925,3346116,5186852,604269
2024,3,11319,11319,793,11297,18583348,0,0,0


In [0]:
from pyspark.sql import functions as F

df_excecoes = (
    df_inep
    .filter(F.col("tipo_dimensao") == 2)
    .filter(
        F.col("sigla_uf").isNull()
        | (F.trim(F.col("sigla_uf")) == "")
        | (F.coalesce(F.col("quantidade_vagas"), F.lit(0)) > 0)
        | (F.coalesce(F.col("quantidade_inscritos"), F.lit(0)) > 0)
    )
)

display(
    df_excecoes.select(
        "ano_censo",
        "sigla_uf",
        "codigo_ies",
        "codigo_curso",
        "nome_curso",
        "codigo_cine_rotulo",
        "tipo_dimensao",
        "quantidade_vagas",
        "quantidade_inscritos",
        "quantidade_ingressantes",
        "quantidade_matriculas",
        "quantidade_concluintes"
    ).orderBy("ano_censo", "sigla_uf")
)

display(
    df_excecoes
    .groupBy("ano_censo")
    .agg(
        F.count("*").alias("registros_excecao"),
        F.sum(
            F.when(
                F.col("sigla_uf").isNull()
                | (F.trim(F.col("sigla_uf")) == ""),
                1
            ).otherwise(0)
        ).alias("registros_sem_uf"),
        F.sum("quantidade_vagas").alias("vagas"),
        F.sum("quantidade_inscritos").alias("inscritos"),
        F.sum("quantidade_ingressantes").alias("ingressantes"),
        F.sum("quantidade_matriculas").alias("matriculas"),
        F.sum("quantidade_concluintes").alias("concluintes")
    )
    .orderBy("ano_censo")
)

ano_censo,sigla_uf,codigo_ies,codigo_curso,nome_curso,codigo_cine_rotulo,tipo_dimensao,quantidade_vagas,quantidade_inscritos,quantidade_ingressantes,quantidade_matriculas,quantidade_concluintes
2022,null,417,5001662,Abi - Educação Física,0011A09,2,0,0,11,8,0
2023,null,1892,5001694,Abi - Educação Física,0011A09,2,0,0,0,0,0
2023,null,417,5001662,Abi - Educação Física,0011A09,2,0,0,13,13,0
2023,null,482,50017101,Abi - Educação Física,0011A09,2,0,0,3,3,0
2023,null,10,5001649,Abi - Educação Física,0011A09,2,0,0,0,0,0
2024,null,953,50017193,Abi - Educação Física,0011A09,2,3702,817,0,0,0
2024,null,10,5001649,Abi - Educação Física,0011A09,2,0,0,0,0,0
2024,null,482,50017101,Abi - Educação Física,0011A09,2,0,0,28,24,0
2024,null,1490,5001691,Abi - Educação Física,0011A09,2,0,0,0,0,0
2024,null,275,50017183,Abi - Educação Física,0011A09,2,809,216,0,0,0


ano_censo,registros_excecao,registros_sem_uf,vagas,inscritos,ingressantes,matriculas,concluintes
2022,1,1,0,0,11,8,0
2023,4,4,0,0,16,16,0
2024,9,9,6925,3590,40,41,0


### Construção da Agregação Gold por UF 
Gold educacional destinada ao EDA e à futura integração com o Caged
**As regras serão:**
- Ingressantes, matrículas e concluintes: dimensões 1 e 2.
- Vagas, inscritos e quantidade de cursos: somente dimensão 1, identificados explicitamente como oferta presencial.
- Registros sem UF: não entram na fato territorial, mas continuam disponíveis na Silver e serão considerados na futura Gold nacional.
- Dimensões 3 e 4: não entram na fato por UF.

In [0]:
from pyspark.sql import functions as F

# Registros com localização territorial válida
df_uf = (
    df_inep
    .filter(F.col("tipo_dimensao").isin(1, 2))
    .filter(
        F.col("sigla_uf").rlike("^[A-Z]{2}$")
    )
)

# Selecionar somente as dimensões necessárias
df_gold_uf = (
    df_uf
    .groupBy(
        "ano_censo",
        "sigla_uf",
        "codigo_cine_area_geral",
        "nome_cine_area_geral",
        "codigo_cine_area_especifica",
        "nome_cine_area_especifica",
        "codigo_cine_area_detalhada",
        "nome_cine_area_detalhada",
        "tipo_modalidade_ensino"
    )
    .agg(
        # Indicadores de estudantes: presencial + EAD
        F.sum("quantidade_ingressantes").alias("ingressantes"),
        F.sum("quantidade_matriculas").alias("matriculas"),
        F.sum("quantidade_concluintes").alias("concluintes"),

        # Indicadores de oferta: somente presencial
        F.sum(
            F.when(
                F.col("tipo_dimensao") == 1,
                F.col("quantidade_cursos")
            ).otherwise(0)
        ).alias("cursos_presenciais"),

        F.sum(
            F.when(
                F.col("tipo_dimensao") == 1,
                F.col("quantidade_vagas")
            ).otherwise(0)
        ).alias("vagas_presenciais"),

        F.sum(
            F.when(
                F.col("tipo_dimensao") == 1,
                F.col("quantidade_inscritos")
            ).otherwise(0)
        ).alias("inscritos_presenciais")
    )
)

print("DataFrame Gold UF preparado.")

display(
    df_gold_uf
    .orderBy(
        "ano_censo",
        "sigla_uf",
        "codigo_cine_area_detalhada"
    )
    .limit(20)
)

DataFrame Gold UF preparado.


ano_censo,sigla_uf,codigo_cine_area_geral,nome_cine_area_geral,codigo_cine_area_especifica,nome_cine_area_especifica,codigo_cine_area_detalhada,nome_cine_area_detalhada,tipo_modalidade_ensino,ingressantes,matriculas,concluintes,cursos_presenciais,vagas_presenciais,inscritos_presenciais
2021,AC,00,Programas básicos,001,Programas básicos,0011,Programas básicos,1,77,57,0,0,263,564
2021,AC,01,Educação,011,Educação,0111,Ciência da educação,1,76,224,39,2,80,406
2021,AC,01,Educação,011,Educação,0111,Ciência da educação,2,35,48,1,0,0,0
2021,AC,01,Educação,011,Educação,0113,Formação de professores sem áreas específicas,2,1189,3467,806,0,0,0
2021,AC,01,Educação,011,Educação,0113,Formação de professores sem áreas específicas,1,145,861,93,5,757,721
2021,AC,01,Educação,011,Educação,0114,Formação de professores em áreas específicas (exceto Letras),2,608,1441,222,0,0,0
2021,AC,01,Educação,011,Educação,0114,Formação de professores em áreas específicas (exceto Letras),1,541,2321,320,23,1467,3788
2021,AC,01,Educação,011,Educação,0115,Formação de professores de letras,1,224,991,140,8,596,1429
2021,AC,01,Educação,011,Educação,0115,Formação de professores de letras,2,283,594,82,0,0,0
2021,AC,01,Educação,018,Programas interdisciplinares abrangendo educação,0188,Programas interdisciplinares abrangendo educação,2,7,9,0,0,0,0


In [0]:
metricas = [
    ("quantidade_ingressantes", "ingressantes"),
    ("quantidade_matriculas", "matriculas"),
    ("quantidade_concluintes", "concluintes")
]

origem = (
    df_uf.groupBy("ano_censo")
    .agg(*[
        F.sum(campo_origem).alias(campo_destino)
        for campo_origem, campo_destino in metricas
    ])
)

destino = (
    df_gold_uf.groupBy("ano_censo")
    .agg(*[
        F.sum(campo_destino).alias(campo_destino)
        for _, campo_destino in metricas
    ])
)

comparacao = origem.alias("b").join(
    destino.alias("g"),
    "ano_censo"
)

for _, campo in metricas:
    comparacao = comparacao.withColumn(
        f"diferenca_{campo}",
        F.col(f"b.{campo}") - F.col(f"g.{campo}")
    )

display(
    comparacao.select(
        "ano_censo",
        *[f"diferenca_{campo}" for _, campo in metricas]
    ).orderBy("ano_censo")
)

ano_censo,diferenca_ingressantes,diferenca_matriculas,diferenca_concluintes
2021,0,0,0
2022,0,0,0
2023,0,0,0
2024,0,0,0


In [0]:
from pyspark.sql import functions as F

chaves_gold = [
    "ano_censo",
    "sigla_uf",
    "codigo_cine_area_detalhada",
    "tipo_modalidade_ensino"
]

# Verificar duplicidades
df_duplicados = (
    df_gold_uf
    .groupBy(*chaves_gold)
    .count()
    .filter(F.col("count") > 1)
)

qtd_duplicados = df_duplicados.count()

print(f"Chaves duplicadas: {qtd_duplicados}")

if qtd_duplicados > 0:
    display(df_duplicados)
    raise ValueError(
        "Existem duplicidades na chave analítica da Gold."
    )

# Validar preenchimento da chave
df_chaves_nulas = df_gold_uf.filter(
    F.col("ano_censo").isNull()
    | F.col("sigla_uf").isNull()
    | F.col("codigo_cine_area_detalhada").isNull()
    | F.col("tipo_modalidade_ensino").isNull()
)

qtd_chaves_nulas = df_chaves_nulas.count()

print(f"Registros com chaves nulas: {qtd_chaves_nulas}")

if qtd_chaves_nulas > 0:
    display(df_chaves_nulas.limit(20))
    raise ValueError(
        "Existem chaves nulas na Gold."
    )

print("Chave analítica validada!")

Chaves duplicadas: 0
Registros com chaves nulas: 0
Chave analítica validada!


In [0]:
TABELA_GOLD_UF = (
    "workspace.gold.fato_inep_educacao_uf_anual"
)

if spark.catalog.tableExists(TABELA_GOLD_UF):
    raise ValueError(
        f"A tabela {TABELA_GOLD_UF} já existe. "
        "Revisar antes de sobrescrever."
    )

df_gold_uf = (
    df_gold_uf
    .withColumn(
        "cursos_presenciais",
        F.when(
            F.col("tipo_modalidade_ensino") == 1,
            F.col("cursos_presenciais")
        )
    )
    .withColumn(
        "vagas_presenciais",
        F.when(
            F.col("tipo_modalidade_ensino") == 1,
            F.col("vagas_presenciais")
        )
    )
    .withColumn(
        "inscritos_presenciais",
        F.when(
            F.col("tipo_modalidade_ensino") == 1,
            F.col("inscritos_presenciais")
        )
    )
)

df_gold_final = (
    df_gold_uf
    .withColumn("dt_processamento", F.current_timestamp())
    .withColumn("versao_transformacao", F.lit("inep_gold_uf_v1"))
)

(
    df_gold_final.write
    .format("delta")
    .mode("error")
    .saveAsTable(TABELA_GOLD_UF)
)

print(f"Tabela criada: {TABELA_GOLD_UF}")
print(f"Registros: {spark.table(TABELA_GOLD_UF).count():,}")

Tabela criada: workspace.gold.fato_inep_educacao_uf_anual
Registros: 14,097


### Gold Inep nacional

In [0]:
from pyspark.sql import functions as F

# Dimensões utilizadas na visão Brasil
df_brasil = df_inep.filter(
    F.col("tipo_dimensao").isin(1, 2, 3)
)

# Preservamos a hierarquia CINE e a modalidade
dimensoes = [
    "ano_censo",
    "codigo_cine_area_geral",
    "nome_cine_area_geral",
    "codigo_cine_area_especifica",
    "nome_cine_area_especifica",
    "codigo_cine_area_detalhada",
    "nome_cine_area_detalhada",
    "tipo_modalidade_ensino"
]

df_gold_brasil = (
    df_brasil
    .groupBy(*dimensoes)
    .agg(
        # Estudantes: presencial (1) e EAD territorializado (2)
        F.sum(
            F.when(
                F.col("tipo_dimensao").isin(1, 2),
                F.col("quantidade_ingressantes")
            ).otherwise(0)
        ).alias("ingressantes"),

        F.sum(
            F.when(
                F.col("tipo_dimensao").isin(1, 2),
                F.col("quantidade_matriculas")
            ).otherwise(0)
        ).alias("matriculas"),

        F.sum(
            F.when(
                F.col("tipo_dimensao").isin(1, 2),
                F.col("quantidade_concluintes")
            ).otherwise(0)
        ).alias("concluintes"),

        # Oferta: presencial (1) e EAD nacional (3)
        F.sum(
            F.when(
                F.col("tipo_dimensao").isin(1, 3),
                F.col("quantidade_cursos")
            ).otherwise(0)
        ).alias("cursos"),

        F.sum(
            F.when(
                F.col("tipo_dimensao").isin(1, 3),
                F.col("quantidade_vagas")
            ).otherwise(0)
        ).alias("vagas"),

        F.sum(
            F.when(
                F.col("tipo_dimensao").isin(1, 3),
                F.col("quantidade_inscritos")
            ).otherwise(0)
        ).alias("inscritos")
    )
)

print("DataFrame nacional preparado.")

display(
    df_gold_brasil
    .orderBy(
        "ano_censo",
        "codigo_cine_area_detalhada",
        "tipo_modalidade_ensino"
    )
    .limit(20)
)

DataFrame nacional preparado.


ano_censo,codigo_cine_area_geral,nome_cine_area_geral,codigo_cine_area_especifica,nome_cine_area_especifica,codigo_cine_area_detalhada,nome_cine_area_detalhada,tipo_modalidade_ensino,ingressantes,matriculas,concluintes,cursos,vagas,inscritos
2021,00,Programas básicos,001,Programas básicos,0011,Programas básicos,1,16751,27631,0,0,27731,115218
2021,00,Programas básicos,001,Programas básicos,0011,Programas básicos,2,322,276,0,0,2760,1235
2021,01,Educação,011,Educação,0111,Ciência da educação,1,182,550,86,5,400,1259
2021,01,Educação,011,Educação,0111,Ciência da educação,2,8356,9839,137,28,46252,17419
2021,01,Educação,011,Educação,0112,Formação de professores de educação infantil,1,40,174,10,2,50,361
2021,01,Educação,011,Educação,0113,Formação de professores sem áreas específicas,1,37678,195710,40858,1535,258473,228411
2021,01,Educação,011,Educação,0113,Formação de professores sem áreas específicas,2,252802,607487,126671,439,787300,560499
2021,01,Educação,011,Educação,0114,Formação de professores em áreas específicas (exceto Letras),1,79635,350498,41608,3531,307414,459265
2021,01,Educação,011,Educação,0114,Formação de professores em áreas específicas (exceto Letras),2,159072,296625,48762,959,1800272,441340
2021,01,Educação,011,Educação,0115,Formação de professores de letras,1,20384,92144,11174,926,68095,117464


In [0]:
metricas = {
    "ingressantes": (
        "quantidade_ingressantes", [1, 2]
    ),
    "matriculas": (
        "quantidade_matriculas", [1, 2]
    ),
    "concluintes": (
        "quantidade_concluintes", [1, 2]
    ),
    "cursos": (
        "quantidade_cursos", [1, 3]
    ),
    "vagas": (
        "quantidade_vagas", [1, 3]
    ),
    "inscritos": (
        "quantidade_inscritos", [1, 3]
    )
}

# Totais esperados calculados na Silver
expressoes = []

for destino, (origem, dimensoes_validas) in metricas.items():
    expressoes.append(
        F.sum(
            F.when(
                F.col("tipo_dimensao").isin(dimensoes_validas),
                F.col(origem)
            ).otherwise(0)
        ).alias(destino)
    )

df_totais_silver = (
    df_brasil
    .groupBy("ano_censo")
    .agg(*expressoes)
)

# Totais produzidos na Gold nacional
df_totais_gold = (
    df_gold_brasil
    .groupBy("ano_censo")
    .agg(
        *[
            F.sum(nome).alias(nome)
            for nome in metricas
        ]
    )
)

# Comparar as seis medidas
df_comparacao = df_totais_silver.alias("s").join(
    df_totais_gold.alias("g"),
    "ano_censo"
)

df_diferencas = df_comparacao.select(
    "ano_censo",
    *[
        (
            F.col(f"s.{nome}") -
            F.col(f"g.{nome}")
        ).alias(f"diferenca_{nome}")
        for nome in metricas
    ]
).orderBy("ano_censo")

display(df_diferencas)

ano_censo,diferenca_ingressantes,diferenca_matriculas,diferenca_concluintes,diferenca_cursos,diferenca_vagas,diferenca_inscritos
2021,0,0,0,0,0,0
2022,0,0,0,0,0,0
2023,0,0,0,0,0,0
2024,0,0,0,0,0,0


In [0]:
chaves_brasil = [
    "ano_censo",
    "codigo_cine_area_detalhada",
    "tipo_modalidade_ensino"
]

# Verificar duplicidades
duplicados = (
    df_gold_brasil
    .groupBy(*chaves_brasil)
    .count()
    .filter(F.col("count") > 1)
)

qtd_duplicados = duplicados.count()

print("Chaves duplicadas:", qtd_duplicados)

if qtd_duplicados > 0:
    display(duplicados.limit(20))

# Verificar chaves nulas
condicao_nulos = (
    F.col("ano_censo").isNull()
    | F.col("codigo_cine_area_detalhada").isNull()
    | F.col("tipo_modalidade_ensino").isNull()
)

qtd_nulos = df_gold_brasil.filter(
    condicao_nulos
).count()

print("Chaves nulas:", qtd_nulos)

if qtd_duplicados > 0 or qtd_nulos > 0:
    raise ValueError(
        "A Gold nacional apresenta inconsistências nas chaves."
    )

print("Chaves da Gold nacional validadas.")

Chaves duplicadas: 0
Chaves nulas: 0
Chaves da Gold nacional validadas.


In [0]:
from pyspark.sql import functions as F

colunas_diferencas = [
    "diferenca_ingressantes",
    "diferenca_matriculas",
    "diferenca_concluintes",
    "diferenca_cursos",
    "diferenca_vagas",
    "diferenca_inscritos"
]

falhas = df_diferencas.filter(
    F.lit(False) |
    (F.coalesce(F.col("diferenca_ingressantes"), F.lit(-1)) != 0) |
    (F.coalesce(F.col("diferenca_matriculas"), F.lit(-1)) != 0) |
    (F.coalesce(F.col("diferenca_concluintes"), F.lit(-1)) != 0) |
    (F.coalesce(F.col("diferenca_cursos"), F.lit(-1)) != 0) |
    (F.coalesce(F.col("diferenca_vagas"), F.lit(-1)) != 0) |
    (F.coalesce(F.col("diferenca_inscritos"), F.lit(-1)) != 0)
).count()

if falhas:
    display(df_diferencas)
    raise ValueError(
        "Divergências identificadas entre Silver e Gold nacional."
    )

print("Todas as diferenças são zero.")

Todas as diferenças são zero.


In [0]:
TABELA_GOLD_BRASIL = (
    "workspace.gold.fato_inep_educacao_brasil_anual"
)

if spark.catalog.tableExists(TABELA_GOLD_BRASIL):
    raise ValueError(
        f"A tabela {TABELA_GOLD_BRASIL} já existe."
    )

df_gold_brasil_final = (
    df_gold_brasil
    .withColumn(
        "dt_processamento",
        F.current_timestamp()
    )
    .withColumn(
        "versao_transformacao",
        F.lit("inep_gold_brasil_v1")
    )
)

(
    df_gold_brasil_final.write
    .format("delta")
    .mode("error")
    .saveAsTable(TABELA_GOLD_BRASIL)
)

print(f"Tabela criada: {TABELA_GOLD_BRASIL}")
print(
    "Registros:",
    spark.table(TABELA_GOLD_BRASIL).count()
)

Tabela criada: workspace.gold.fato_inep_educacao_brasil_anual
Registros: 653


In [0]:
df_gold_persistida = spark.table(TABELA_GOLD_BRASIL)

display(
    df_gold_persistida
    .groupBy("ano_censo", "tipo_modalidade_ensino")
    .agg(
        F.count("*").alias("registros"),
        F.sum("ingressantes").alias("ingressantes"),
        F.sum("matriculas").alias("matriculas"),
        F.sum("concluintes").alias("concluintes"),
        F.sum("cursos").alias("cursos"),
        F.sum("vagas").alias("vagas"),
        F.sum("inscritos").alias("inscritos")
    )
    .orderBy("ano_censo", "tipo_modalidade_ensino")
)

ano_censo,tipo_modalidade_ensino,registros,ingressantes,matriculas,concluintes,cursos,vagas,inscritos
2021,1,88,1467717,5270750,842184,35482,5942212,8644865
2021,2,74,2476016,3713646,484777,7620,16736850,6475962
2022,1,88,1656401,5113182,803801,35774,5658590,8811102
2022,2,74,3098768,4328116,483335,9186,17171895,8176586
2023,1,89,1679790,5063936,783505,35410,5440999,8627123
2023,2,75,3312712,4910515,590782,10554,19180966,5887566
2024,1,87,1663040,5037875,729246,34479,5075146,8658561
2024,2,78,3346116,5186852,604269,11297,18583348,7060934
